In [1]:
# ==========================================
# 1. УСТАНОВКА БИБЛИОТЕК И НАСТРОЙКИ
# ==========================================
!pip install transformers torch scikit-learn razdel rouge tqdm

import os
import glob
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from torch.optim import AdamW

from sklearn.model_selection import train_test_split
from rouge import Rouge
from razdel import sentenize
import numpy as np
import random
from tqdm.auto import tqdm

# Подключаем Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Фиксация Random Seed для воспроизводимости
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

# КОНФИГУРАЦИЯ
CONFIG = {
    "model_name": "cointegrated/rubert-tiny",
    "data_path_texts": "/content/drive/MyDrive/Summarization_Project/texts",     # Путь к полным текстам
    "data_path_summaries": "/content/drive/MyDrive/Summarization_Project/summaries", # Путь к саммари
    "max_len": 64,          # Длина одного предложения (для tiny можно меньше 128)
    "batch_size": 16,
    "epochs": 4,            # Согласно плану
    "learning_rate": 2e-5,
    "device": torch.device("cuda" if torch.cuda.is_available() else "cpu"),
    "target_rouge": 0.4
}

print(f"Device: {CONFIG['device']}")

# ==========================================
# 2. ФУНКЦИИ ЗАГРУЗКИ И ОБРАБОТКИ ДАННЫХ
# ==========================================

def read_file(path):
    """Читает текстовый файл с учетом кодировки."""
    try:
        with open(path, 'r', encoding='utf-8') as f:
            return f.read().strip()
    except UnicodeDecodeError:
        # Fallback для windows-1251, если вдруг попадется
        with open(path, 'r', encoding='cp1251') as f:
            return f.read().strip()

def split_sentences(text):
    """Разбивает текст на предложения с помощью razdel."""
    return [s.text for s in sentenize(text)]

def get_greedy_match(book_sents, summary_sents, threshold=0.55):
    """
    Алгоритм создания "Золотого стандарта" (Oracle summary).
    Для каждого предложения из саммари ищет наиболее похожее предложение в книге
    и помечает его как '1' (важное).
    """
    labels = [0] * len(book_sents)
    rouge = Rouge()

    # Для ускорения можно использовать Jaccard similarity вместо ROUGE на этапе препроцессинга
    # Здесь упрощенная реализация через пересечение токенов (быстрее)

    summary_tokens = [set(s.lower().split()) for s in summary_sents]
    book_tokens = [set(s.lower().split()) for s in book_sents]

    for summ_tok in summary_tokens:
        best_idx = -1
        best_score = 0.0

        if len(summ_tok) < 3: continue # Пропускаем слишком короткие

        for idx, book_tok in enumerate(book_tokens):
            if len(book_tok) < 3: continue

            # Коэффициент Жаккара (пересечение / объединение)
            intersection = len(summ_tok.intersection(book_tok))
            union = len(summ_tok.union(book_tok))
            score = intersection / union if union > 0 else 0

            if score > best_score:
                best_score = score
                best_idx = idx

        # Если нашли достаточно похожее предложение, ставим метку 1
        if best_idx != -1 and best_score > 0.2: # Порог схожести (можно настраивать)
            labels[best_idx] = 1

    return labels

def load_dataset_from_drive(texts_dir, summaries_dir):
    """Загружает пары (книга, саммари) из папок."""
    data = []

    # Получаем список файлов
    text_files = sorted(glob.glob(os.path.join(texts_dir, "*.txt")))

    print(f"Найдено книг: {len(text_files)}")

    for text_path in tqdm(text_files, desc="Чтение и разметка книг"):
        filename = os.path.basename(text_path)
        summary_path = os.path.join(summaries_dir, filename)

        if os.path.exists(summary_path):
            full_text = read_file(text_path)
            summary_text = read_file(summary_path)

            # Разбиваем на предложения
            book_sents = split_sentences(full_text)
            summ_sents = split_sentences(summary_text)

            # Генерируем метки (1/0)
            labels = get_greedy_match(book_sents, summ_sents)

            # Добавляем в список данных
            # Чтобы не забивать память, сохраняем сразу парами (предложение, метка)
            for s, l in zip(book_sents, labels):
                data.append({"text": s, "label": l})
        else:
            print(f"Внимание: Не найдено саммари для {filename}")

    return data

# ==========================================
# 3. КЛАСС ДАТАСЕТА И МОДЕЛЬ
# ==========================================

class SummarizationDataset(Dataset):
    def __init__(self, data, tokenizer, max_len):
        self.data = data
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        text = str(item['text'])
        label = item['label']

        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.float)
        }

class BertExtractiveSummarizer(nn.Module):
    def __init__(self, model_name):
        super(BertExtractiveSummarizer, self).__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(0.3)
        # Линейный слой классификации: 312 -> 1
        self.classifier = nn.Linear(self.bert.config.hidden_size, 1)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        # Берем вектор [CLS] токена или pooled output
        # Для tiny rubert output.pooler_output может отсутствовать, берем первый токен
        cls_output = outputs.last_hidden_state[:, 0, :]
        x = self.dropout(cls_output)
        x = self.classifier(x)
        return x

# ==========================================
# 4. ЦИКЛ ОБУЧЕНИЯ (TRAIN LOOP)
# ==========================================

def train_epoch(model, data_loader, optimizer, scheduler, device, loss_fn):
    model = model.train()
    losses = []

    for batch in tqdm(data_loader, desc="Training", leave=False):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        targets = batch["labels"].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.squeeze() # [batch_size]

        # Если batch_size=1, нужно обработать размерность
        if logits.dim() == 0: logits = logits.unsqueeze(0)

        loss = loss_fn(logits, targets)

        losses.append(loss.item())

        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

    return np.mean(losses)

# Функция для генерации итогового саммари
def generate_summary(text, model, tokenizer, top_n=5):
    model.eval()
    sentences = split_sentences(text)
    scores = []

    # Прогоняем предложения через модель (можно батчами, здесь поштучно для простоты)
    for sent in sentences:
        inputs = tokenizer.encode_plus(
            sent, return_tensors='pt', max_length=CONFIG['max_len'],
            padding='max_length', truncation=True
        )
        input_ids = inputs['input_ids'].to(CONFIG['device'])
        mask = inputs['attention_mask'].to(CONFIG['device'])

        with torch.no_grad():
            logits = model(input_ids, mask)
            prob = torch.sigmoid(logits).item()
            scores.append((sent, prob))

    # Сортируем: сначала берем с самой высокой вероятностью
    top_sentences = sorted(scores, key=lambda x: x[1], reverse=True)[:top_n]

    # ВАЖНО: Для читаемости восстанавливаем исходный порядок предложений
    # Чтобы понять порядок, нам нужно найти индексы этих предложений в исходном списке
    final_summary_sents = []
    for sent in sentences:
        # Если предложение есть в топе
        if any(s[0] == sent for s in top_sentences):
            final_summary_sents.append(sent)

    return " ".join(final_summary_sents)

# ==========================================
# 5. ОСНОВНОЙ ЗАПУСК
# ==========================================

# А. Загрузка данных
print("--- Старт загрузки данных ---")
raw_data = load_dataset_from_drive(CONFIG['data_path_texts'], CONFIG['data_path_summaries'])

if len(raw_data) == 0:
    print("ОШИБКА: Данные не найдены. Проверьте пути на Google Drive!")
else:
    print(f"Всего предложений для обучения: {len(raw_data)}")

    # Б. Подготовка DataLoader
    tokenizer = AutoTokenizer.from_pretrained(CONFIG['model_name'])
    dataset = SummarizationDataset(raw_data, tokenizer, CONFIG['max_len'])

    # 90% на обучение, 10% на валидацию (сплит по предложениям)
    train_data, val_data = train_test_split(dataset, test_size=0.1, random_state=42)

    train_loader = DataLoader(train_data, batch_size=CONFIG['batch_size'], shuffle=True)
    val_loader = DataLoader(val_data, batch_size=CONFIG['batch_size'])

    # В. Инициализация модели
    model = BertExtractiveSummarizer(CONFIG['model_name'])
    model = model.to(CONFIG['device'])

    # Вес для класса '1' (так как важных предложений меньше, чем мусора)
    # Примерно 1 к 10, поэтому вес положительного класса ставим выше
    pos_weight = torch.tensor([5.0]).to(CONFIG['device'])
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    optimizer = AdamW(model.parameters(), lr=CONFIG['learning_rate'])
    total_steps = len(train_loader) * CONFIG['epochs']
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)

    # Г. Обучение
    print(f"\n--- Старт обучения на {CONFIG['epochs']} эпох ---")
    for epoch in range(CONFIG['epochs']):
        train_loss = train_epoch(model, train_loader, optimizer, scheduler, CONFIG['device'], loss_fn)
        print(f"Epoch {epoch+1}/{CONFIG['epochs']} | Loss: {train_loss:.4f}")

        # Сохранение чекпоинта
        torch.save(model.state_dict(), f"/content/drive/MyDrive/Summarization_Project/bert_summ_ep{epoch+1}.bin")

    print("\nОбучение завершено. Чекпоинты сохранены на Диске.")

    # Д. Оценка (Валидация на тестовом тексте)
    # Берем случайный текст из загруженных файлов для демонстрации
    test_files = glob.glob(os.path.join(CONFIG['data_path_texts'], "*.txt"))
    if test_files:
        test_text_path = test_files[0]
        ref_path = os.path.join(CONFIG['data_path_summaries'], os.path.basename(test_text_path))

        full_text = read_file(test_text_path)
        reference = read_file(ref_path)

        # Генерируем (берем топ-5 предложений)
        generated = generate_summary(full_text, model, tokenizer, top_n=5)

        print("\n=== РЕЗУЛЬТАТ ===")
        print("Сгенерированное саммари:\n", generated[:500], "...") # Показываем начало

        # Считаем ROUGE
        rouge = Rouge()
        try:
            scores = rouge.get_scores(generated, reference)[0]
            print(f"\nROUGE-L F1: {scores['rouge-l']['f']:.4f}")
            if scores['rouge-l']['f'] > CONFIG['target_rouge']:
                print("Цель достигнута!")
            else:
                print(f"Цель {CONFIG['target_rouge']} пока не достигнута.")
        except Exception as e:
            print("Ошибка при подсчете ROUGE (возможно пустой вывод):", e)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Device: cuda
--- Старт загрузки данных ---
Найдено книг: 6


Чтение и разметка книг:   0%|          | 0/6 [00:00<?, ?it/s]

Всего предложений для обучения: 31195


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(



--- Старт обучения на 4 эпох ---


...

Epoch 2/4 | Loss: 0.1919


Training:   0%|          | 0/1755 [00:00<?, ?it/s]

Epoch 3/4 | Loss: 0.1734


Training:   0%|          | 0/1755 [00:00<?, ?it/s]

Epoch 4/4 | Loss: 0.1642

Обучение завершено. Чекпоинты сохранены на Диске.

=== РЕЗУЛЬТАТ ===
Сгенерированное саммари:
 Петруша в Петербург не поедет. Глава III Крепость

Белогорская крепость находилась в сорока верстах от Оренбурга. Марья Ивановна села в угол и стала шить. Он встал и вышел из комнаты. Вскоре потом Петр Андреевич женился на Марье Ивановне. ...

ROUGE-L F1: 0.0609
Цель 0.4 пока не достигнута.


In [9]:
import os
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel
from razdel import sentenize
import math

# ==========================================
# 1. НАСТРОЙКИ
# ==========================================
# Укажите файл (короткий рассказ или целая книга)
FILENAME = "test.txt"

# Пути
BASE_PATH = "/content/drive/MyDrive/Summarization_Project"
MODEL_PATH = os.path.join(BASE_PATH, "bert_summ_ep4.bin")
TEXTS_PATH = os.path.join(BASE_PATH)
MODEL_NAME = "cointegrated/rubert-tiny"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ==========================================
# 2. ИНИЦИАЛИЗАЦИЯ МОДЕЛИ
# ==========================================
class BertExtractiveSummarizer(nn.Module):
    def __init__(self, model_name):
        super(BertExtractiveSummarizer, self).__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(self.bert.config.hidden_size, 1)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :]
        x = self.dropout(cls_output)
        return self.classifier(x)

print("⏳ Загрузка модели...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = BertExtractiveSummarizer(MODEL_NAME)

if os.path.exists(MODEL_PATH):
    model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
    model.to(DEVICE)
    model.eval()
    print("Модель успешно загружена из файла!")
else:
    raise FileNotFoundError("Веса модели не найдены!")

# ==========================================
# 3. УМНАЯ ЛОГИКА (ADAPTIVE + GLUE)
# ==========================================

def rate_sentences(sentences, model, tokenizer):
    """Присваивает каждому предложению оценку важности (0..1)."""
    scores = []
    for i, sent in enumerate(sentences):
        # Пропускаем совсем мусор (слишком короткие < 15 символов)
        if len(sent) < 15:
            # Добавляем с нулевым скором, чтобы не сбивать индексацию
            scores.append((sent, 0.0, i))
            continue

        try:
            inputs = tokenizer.encode_plus(
                sent, return_tensors='pt', max_length=128,
                padding='max_length', truncation=True
            )
            input_ids = inputs['input_ids'].to(DEVICE)
            mask = inputs['attention_mask'].to(DEVICE)

            with torch.no_grad():
                logits = model(input_ids, mask)
                score = torch.sigmoid(logits).item()
                scores.append((sent, score, i))
        except:
             scores.append((sent, 0.0, i))
    return scores

def summarize_adaptive(filename):
    input_file = os.path.join(TEXTS_PATH, filename)
    if not os.path.exists(input_file):
        print("Файл не найден!")
        return

    print(f"Читаю {filename}...")
    with open(input_file, 'r', encoding='utf-8') as f:
        text = f.read().replace('\n', ' ')

    all_sentences = [s.text for s in sentenize(text)]
    total_sents = len(all_sentences)
    print(f"Всего предложений: {total_sents}")

    final_summary_sentences = []

    # Считаем скоры
    scores = rate_sentences(all_sentences, model, tokenizer)

    # === ЛОГИКА ДЛЯ КОРОТКОГО ТЕКСТА ===
    if total_sents < 100:
        print("⚡ Режим: Short Story (Buckets + Smoothing)")

        # Цель: ~25% текста
        target_count = max(3, int(total_sents * 0.25))
        bucket_size = math.ceil(total_sents / target_count)

        selected_indices = set()

        # 1. Всегда берем ПЕРВОЕ предложение (экспозиция)
        print("   -> Добавлено вступление (sentence #0)")
        selected_indices.add(0)

        # 2. Секторальное разделение (как раньше)
        for i in range(target_count):
            start_idx = i * bucket_size
            end_idx = min((i + 1) * bucket_size, total_sents)
            if start_idx >= total_sents: break

            sector_scores = scores[start_idx:end_idx]
            valid_candidates = [s for s in sector_scores if s[1] > 0]

            if valid_candidates:
                best_in_sector = max(valid_candidates, key=lambda x: x[1])
                selected_indices.add(best_in_sector[2])

        # 3. Умная склейка (предыдущий контекст)
        # Делаем копию, чтобы итерироваться
        current_indices = sorted(list(selected_indices))
        for idx in current_indices:
            if idx > 0:
                prev_idx = idx - 1
                # Если предыдущее длинное и не выбрано -> берем
                if prev_idx not in selected_indices and len(all_sentences[prev_idx]) > 20:
                    # Дополнительная эвристика: если текущее начинается с "Она", "Он", "Мы" - точно нужен контекст
                    first_word = all_sentences[idx].split()[0].lower()
                    if first_word in ['она', 'он', 'мы', 'они', 'я']:
                        print(f"   -> Контекст (местоимение): к №{idx} добавлено №{prev_idx}")
                        selected_indices.add(prev_idx)

        # 4. === НОВОЕ: GAP FILLING (Сглаживание) ===
        # Если между выбранными предложениями дырка всего в 1-2 предложения,
        # забираем их тоже, чтобы не терять нить повествования.
        sorted_indices = sorted(list(selected_indices))
        final_indices = set(selected_indices)

        for i in range(len(sorted_indices) - 1):
            current = sorted_indices[i]
            next_val = sorted_indices[i+1]
            gap = next_val - current - 1

            # Если пропущено 1 или 2 предложения, забираем их
            if 0 < gap <= 2:
                for fill_idx in range(current + 1, next_val):
                    print(f"   -> Заделка шва: добавлено пропущенное №{fill_idx}")
                    final_indices.add(fill_idx)

        # Финальная сборка
        final_summary_sentences = [all_sentences[i] for i in sorted(list(final_indices))]

    else:
        # Режим книги (без изменений)
        print("Режим: Книга")
        CHUNK_SIZE = 50
        TOP_N_PER_CHUNK = 2
        for i in range(0, total_sents, CHUNK_SIZE):
            chunk_scores = scores[i : i + CHUNK_SIZE]
            top_items = sorted(chunk_scores, key=lambda x: x[1], reverse=True)[:TOP_N_PER_CHUNK]
            final_summary_sentences.extend([x[0] for x in top_items])

    # СБОРКА И СОХРАНЕНИЕ
    result_text = " ".join(final_summary_sentences)

    output_filename = f"SMART_SUMMARY_{filename}"
    save_path = os.path.join(BASE_PATH, output_filename)

    with open(save_path, "w", encoding="utf-8") as f:
        f.write(f"Источник: {filename}\n")
        f.write(f"Режим: {'Short+Glue' if total_sents < 100 else 'Novel'}\n")
        f.write("-" * 20 + "\n")
        f.write(result_text)

    print(f"Готово! Результат в файле: {output_filename}")
    print("\n--- ПРЕДПРОСМОТР РЕЗУЛЬТАТА ---")
    print(result_text[:1000])

# ==========================================
# ЗАПУСК
# ==========================================
summarize_adaptive(FILENAME)

⏳ Загрузка модели...
Модель успешно загружена из файла!
Читаю test.txt...
Всего предложений: 21
⚡ Режим: Short Story (Buckets + Smoothing)
   -> Добавлено вступление (sentence #0)
   -> Контекст (местоимение): к №8 добавлено №7
   -> Контекст (местоимение): к №18 добавлено №17
   -> Контекст (местоимение): к №20 добавлено №19
   -> Заделка шва: добавлено пропущенное №5
   -> Заделка шва: добавлено пропущенное №6
Готово! Результат в файле: SMART_SUMMARY_test.txt

--- ПРЕДПРОСМОТР РЕЗУЛЬТАТА ---
На   Чёрном   озере         Однажды мы ночевали на Чёрном озере, в высоких зарослях, около большой кучи старого хвороста. Рыба нырнула и прошла под резиновой лодкой. Лодка закачалась. Рыба вынырнула снова. Должно быть, это была гигантская щука. Она могла задеть резиновую лодку пером и распороть её, как бритвой. Рыба всё время шла рядом с лодкой. Я бросил в волчицу тяжёлым свинцовым грузилом. Она отскочила и рысцой побежала от берега. И мы увидели, как она пролезла вместе с волчатами в круглую нор